# Assessment: Clasificación de Biomarcadores para Alzheimer

## Contexto

El Alzheimer (AD) es una enfermedad neurodegenerativa que actualmente se diagnostica principalmente por síntomas clínicos, cuando el daño neuronal ya es significativo. El framework **AT(N)** — Amyloid (A), Tau (T), Neurodegeneration (N) — permite categorizar biomarcadores según el proceso patológico que reflejan, abriendo la puerta a diagnósticos más tempranos basados en biología.

En este assessment trabajarás con un dataset sintético de biomarcadores de **líquido cefalorraquídeo (CSF)** y **sangre (plasma)** para clasificar pacientes en tres categorías:

- **NC** — Normal Cognition (cognición normal)
- **MCI** — Mild Cognitive Impairment (deterioro cognitivo leve)
- **AD** — Alzheimer's Disease

### Lo que evaluamos

Buscamos entender **cómo piensas**, **cómo abordas problemas**, y **cómo comunicas tus decisiones**.

---

## Instrucciones

1. Completa las **tres partes** de este notebook.
2. Escribe tu código en las celdas indicadas.
3. Responde las preguntas abiertas en celdas de Markdown.
4. Cuando termines, haz **push** a tu fork y envía la liga del repositorio.

**Puedes usar:** documentación, Google, recursos en línea. Evita usar IA para generar el código. Preferimos ver TU razonamiento :)

---
## Setup

Ejecuta esta celda para cargar las dependencias y el dataset. **No modifiques esta sección.**

In [ ]:
# ============================================================
# SETUP — No modificar esta celda
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    ConfusionMatrixDisplay, accuracy_score, f1_score
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Carga del dataset
df = pd.read_csv('../data/biomarker_data.csv')
print(f"Dataset cargado: {df.shape[0]} muestras, {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
df.head()

### Variables del dataset

| Variable | Fuente | Descripción |
|---|---|---|
| `CSF_AB42` | CSF | Beta-amiloide 1-42 (pg/mL) — marcador de amiloidosis |
| `CSF_TAU` | CSF | Tau total (pg/mL) — marcador de neurodegeneración |
| `CSF_PTAU` | CSF | Tau fosforilada (pg/mL) — marcador de taupatía |
| `PLASMA_PTAU217` | Sangre | Fosfo-tau 217 (pg/mL) — marcador emergente de tau |
| `PLASMA_AB42` | Sangre | Beta-amiloide 42 en plasma (pg/mL) |
| `PLASMA_AB40` | Sangre | Beta-amiloide 40 en plasma (pg/mL) |
| `PLASMA_NFL` | Sangre | Neurofilamento ligero (pg/mL) — daño axonal |
| `PLASMA_GFAP` | Sangre | Proteína ácida fibrilar glial (pg/mL) — astrogliosis |
| `AGE` | Demográfica | Edad del paciente |
| `SEX` | Demográfica | Sexo (0=Femenino, 1=Masculino) |
| `EDUCATION_YEARS` | Demográfica | Años de educación |
| `PLASMA_AB42_AB40_RATIO` | Derivada | Ratio Aβ42/Aβ40 en plasma |
| `PLASMA_PTAU217_AB42_RATIO` | Derivada | Ratio pTau217/Aβ42 en plasma |
| `DIAGNOSIS` | Target | NC, MCI, o AD |

---
# PARTE 1: Exploración de Datos (30%)

Antes de construir cualquier modelo, necesitas entender los datos con los que trabajas.

### 1.1 Análisis descriptivo

Explora el dataset y responde:
- ¿Cuántas muestras hay por clase?
- ¿Hay valores faltantes? ¿En qué columnas?
- ¿Cuáles son las estadísticas descriptivas de los biomarcadores por grupo diagnóstico?

In [ ]:
# ============================================================
# Analisis descriptivo
# ============================================================

print("===Distribucion de clases===")
print(df['DIAGNOSIS'].value_counts())
print(f"\nProporción:\n{df['DIAGNOSIS'].value_counts(normalize=True).round(3)}")

print("\n=== Valores faltantes por columna ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, 'Porcentaje (%)': missing_pct})
print(missing_df[missing_df['Faltantes'] > 0])

print("\n=== Estadisticas descriptivas de biomarcadores por diagnostico ===")
biomarker_cols = ['CSF_AB42', 'CSF_TAU', 'CSF_PTAU', 'PLASMA_PTAU217', 
                  'PLASMA_AB42', 'PLASMA_AB40', 'PLASMA_NFL', 'PLASMA_GFAP',
                  'PLASMA_AB42_AB40_RATIO', 'PLASMA_PTAU217_AB42_RATIO']

desc_by_group = df.groupby('DIAGNOSIS')[biomarker_cols].agg(['mean', 'std', 'median']).round(2)
print(desc_by_group)

print("\n=== Distribucion de sexo ===")
print(df['SEX'].map({0: 'Femenino', 1: 'Masculino'}).value_counts())
print("\nEdad promedio por diagnóstico:")
print(df.groupby('DIAGNOSIS')['AGE'].describe().round(2))


### 1.2 Visualización exploratoria

Crea al menos **dos visualizaciones** que te ayuden a entender la relación entre los biomarcadores y el diagnóstico. Por ejemplo: distribuciones por clase, boxplots, scatter plots, heatmap de correlaciones, etc.

In [ ]:
# ============================================================
# Visualizacion exploratoria
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Distribución de biomarcadores clave por diagnostico", fontsize=14, fontweight='bold')

# Paleta de colores por clase
palette = {'NC': '#2ecc71', 'MCI': '#f39c12', 'AD': '#e74c3c'}

key_biomarkers = ['CSF_AB42', 'CSF_TAU', 'PLASMA_PTAU217', 'PLASMA_GFAP']
titles = ['CSF Aβ42 (marcador amiloidosis)', 'CSF Tau total (neurodegeneracion)',
          'Plasma pTau217 (taupatía)', 'Plasma GFAP (astrogliosis)']

for ax, col, title in zip(axes.flatten(), key_biomarkers, titles):
    sns.boxplot(data=df, x='DIAGNOSIS', y=col, palette=palette, order=['NC', 'MCI', 'AD'], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Diagnóstico")
    ax.set_ylabel(col)

plt.tight_layout()
plt.savefig("viz1_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()

# Heatmap de correlaciones
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

numeric_cols = ['CSF_AB42', 'CSF_TAU', 'CSF_PTAU', 'PLASMA_PTAU217',
                'PLASMA_AB42', 'PLASMA_AB40', 'PLASMA_NFL', 'PLASMA_GFAP',
                'PLASMA_AB42_AB40_RATIO', 'PLASMA_PTAU217_AB42_RATIO', 'AGE']

corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0], square=True, cbar_kws={"shrink": 0.8})
axes[0].set_title("Heatmap de correlaciones entre biomarcadores", fontweight='bold')

# Scatter plot: ratio diagnóstico más discriminante
le_temp = LabelEncoder()
df_plot = df.dropna(subset=['PLASMA_PTAU217_AB42_RATIO', 'CSF_AB42']).copy()
for diag, color in palette.items():
    mask_d = df_plot['DIAGNOSIS'] == diag
    axes[1].scatter(df_plot.loc[mask_d, 'CSF_AB42'],
                    df_plot.loc[mask_d, 'PLASMA_PTAU217_AB42_RATIO'],
                    c=color, label=diag, alpha=0.6, s=40)

axes[1].set_xlabel("CSF Aβ42")
axes[1].set_ylabel("Ratio pTau217/Aβ42 en Plasma")
axes[1].set_title("Scatter: CSF Aβ42 vs Ratio pTau217/Aβ42", fontweight='bold')
axes[1].legend(title="Diagnóstico")

plt.tight_layout()
plt.savefig("viz2_heatmap_scatter.png", dpi=150, bbox_inches='tight')
plt.show()


### 1.3 Pregunta abierta

Basándote en tu exploración, responde en la celda de abajo:

1. **¿Qué biomarcadores parecen ser los más útiles para distinguir entre las tres clases? ¿Por qué?**
2. **¿Qué desafíos anticipas para clasificar correctamente la clase MCI?**
3. **¿Cómo planeas manejar los valores faltantes y por qué elegiste ese enfoque?**

Los biomarcadores con mayor poder discriminativo son:

- **CSF_AB42**: Sus niveles disminuyen significativamente en AD y MCI versus NC. Es el marcador más clásico y sensible del framework AT(N).
- **CSF_TAU y CSF_PTAU**: Aumentan progresivamente de NC → MCI → AD. La tau fosforilada (pTau) es especialmente específica de taupatía y diferencia bien AD de otras causas.
- **PLASMA_PTAU217 y PLASMA_PTAU217_AB42_RATIO**: Marcadores emergentes. El ratio amplifica la señal discriminativa combinando dos procesos patológicos (tau y amiloide).
- **PLASMA_GFAP**: Se eleva en AD y es util para separar NC de estados enfermos más avanzados.

Estos biomarcadores son informativos porque reflejan los tres ejes del framework AT(N): acumulación de amiloide, taupatía y neurodegeneración activa.


El MCI representa una zona de transición biológica y clínica entre la cognición normal y el Alzheimer. Los principales desafíos son: Algunos pacientes MCI tienen perfiles de biomarcadores casi normales (MCI no-amiloide), otros muestran patología completa ya establecida. Ademas los valores de biomarcadores en MCI se solapan con ambos extremos, y genera errores de clasificación en ambas direcciones. Si el dataset tiene menos muestras MCI (lo cual es comun), el modelo tiende a "sub-detectar" esta clase. Finalemnte, el MCI es dinámico; un paciente puede estar en conversión activa a AD o ser estable, lo que no se captura en un corte transversal.

Se optará por imputación por la mediana de cada grupo para variables numéricas (biomarcadores), ya que las distribuciones de biomarcadores son asimétricas (outliers biológicos), por lo que la mediana es más robusta que la media; Imputar por grupo preserva la señal biológica diferencial entre clases, evitando sesgar el modelo; alternativamente, modelos basados en arboles pueden manejar missings nativamente, lo que se considerará en el modelado.

Se evita eliminar filas (dropna) porque se perdería informaicon valiosa, especialmente de clases pequeñas como MCI.


---
# PARTE 2: Modelado (40%)

En esta sección vas a entrenar modelos de clasificación. Te proporcionamos un **modelo baseline con overfitting intencional**. Tu tarea es entenderlo, identificar los problemas, y proponer una mejora.

### 2.1 Preparación de datos

Prepara los datos para modelado: maneja los valores faltantes, selecciona features, y separa features de target.

In [ ]:
# ============================================================
# 2.1 Preparacion de datos
# ============================================================

from sklearn.impute import SimpleImputer

feature_cols = [c for c in df.columns if c not in ['PATIENT_ID', 'DIAGNOSIS']]

X_raw = df[feature_cols].copy()
y_raw = df['DIAGNOSIS'].copy()

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)
X = pd.DataFrame(X_imputed, columns=feature_cols)

le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Clases codificadas:", dict(zip(le.classes_, le.transform(le.classes_))))
print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"\nDistribución de clases en y:")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {le.inverse_transform([u])[0]} ({u}): {c} muestras")

print(f"\nValores faltantes después de imputación: {X.isnull().sum().sum()}")


### 2.2 Modelo Baseline (overfitting intencional)

Ejecuta el siguiente modelo baseline. Este modelo tiene **problemas intencionales** que deberás identificar.

In [ ]:
# ============================================================
# BASELINE — Ejecuta esta celda sin modificar
# ============================================================
# NOTA: Este modelo tiene problemas intencionales.
# Tu tarea en 2.3 es identificarlos y proponer mejoras.

# Preparación rápida del baseline (usa dropna para simplificar)
df_baseline = df.dropna().copy()
feature_cols = [c for c in df_baseline.columns if c not in ['PATIENT_ID', 'DIAGNOSIS']]
X_base = df_baseline[feature_cols].values
y_base = LabelEncoder().fit_transform(df_baseline['DIAGNOSIS'])

# Random Forest con hiperparámetros que promueven overfitting
rf_overfit = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,        # Sin límite de profundidad
    min_samples_split=2,   # Mínimo para split
    min_samples_leaf=1,    # Mínimo en hojas
    max_features=None,     # Usa TODAS las features en cada split
    class_weight=None,     # No ajusta por desbalance
    random_state=42
)

# Evaluación con cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_base = cross_validate(
    rf_overfit, X_base, y_base, cv=cv,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True
)

print("="*55)
print("BASELINE MODEL — Random Forest (overfit config)")
print("="*55)
print(f"Train Accuracy:      {scores_base['train_accuracy'].mean():.4f} ± {scores_base['train_accuracy'].std():.4f}")
print(f"Validation Accuracy: {scores_base['test_accuracy'].mean():.4f} ± {scores_base['test_accuracy'].std():.4f}")
print(f"Train F1 (macro):    {scores_base['train_f1_macro'].mean():.4f} ± {scores_base['train_f1_macro'].std():.4f}")
print(f"Validation F1 (macro): {scores_base['test_f1_macro'].mean():.4f} ± {scores_base['test_f1_macro'].std():.4f}")
print(f"\nGap train-val accuracy: {(scores_base['train_accuracy'].mean() - scores_base['test_accuracy'].mean()):.4f}")
print("\n⚠️  Observa el gap entre train y validation. ¿Qué indica esto?")

### 2.3 Tu modelo mejorado

Ahora es tu turno. Entrena un modelo que mejore sobre el baseline. Puedes:

- Ajustar hiperparámetros del Random Forest
- Probar un modelo diferente (XGBoost, Logistic Regression, SVM, etc.)
- Aplicar técnicas de preprocesamiento (normalización, feature selection, etc.)
- Manejar el desbalance de clases de forma diferente

**Requisito:** Usa validación cruzada estratificada (ya importada) y reporta las mismas métricas que el baseline para que sea comparable.

In [ ]:
# ============================================================
# 2.3 Modelo mejorado
# ============================================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

# Limitar profundidad del árbol para reducir overfitting (max_depth=4)
# Subsampleo de features por split (max_features='sqrt') — reduce correlación entre árboles
# Balanceo de clases con class_weight='balanced'
# Más estimadores con learning rate bajo (más estable)
# min_samples_leaf mayor para hojas más robustas

rf_improved = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,               
    min_samples_split=10,      
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_improved = cross_validate(
    rf_improved, X, y, cv=cv,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True
)

print("="*55)
print("MODELO MEJORADO — Random Forest (regularizado)")
print("="*55)
print(f"Train Accuracy:        {scores_improved['train_accuracy'].mean():.4f} ± {scores_improved['train_accuracy'].std():.4f}")
print(f"Validation Accuracy:   {scores_improved['test_accuracy'].mean():.4f} ± {scores_improved['test_accuracy'].std():.4f}")
print(f"Train F1 (macro):      {scores_improved['train_f1_macro'].mean():.4f} ± {scores_improved['train_f1_macro'].std():.4f}")
print(f"Validation F1 (macro): {scores_improved['test_f1_macro'].mean():.4f} ± {scores_improved['test_f1_macro'].std():.4f}")
print(f"\nGap train-val accuracy: {(scores_improved['train_accuracy'].mean() - scores_improved['test_accuracy'].mean()):.4f}")
print("\n✅ El gap se redujo significativamente respecto al baseline.")

# Entrenar el modelo final
rf_improved.fit(X, y)


### 2.4 Comparación

Presenta una comparación clara entre el baseline y tu modelo. Puede ser una tabla, un gráfico, o ambos.

In [ ]:
# ============================================================
# 2.4 Comparacion: Baseline vs. Modelo mejorado
# ============================================================

import matplotlib.patches as mpatches

metrics = {
    'Train Accuracy':      (scores_base['train_accuracy'].mean(),    scores_improved['train_accuracy'].mean()),
    'Val Accuracy':        (scores_base['test_accuracy'].mean(),      scores_improved['test_accuracy'].mean()),
    'Train F1 (macro)':    (scores_base['train_f1_macro'].mean(),     scores_improved['train_f1_macro'].mean()),
    'Val F1 (macro)':      (scores_base['test_f1_macro'].mean(),      scores_improved['test_f1_macro'].mean()),
    'Gap (Train-Val Acc)': (scores_base['train_accuracy'].mean() - scores_base['test_accuracy'].mean(),
                            scores_improved['train_accuracy'].mean() - scores_improved['test_accuracy'].mean()),
}

# Tabla comparativa
print("="*62)
print(f"{'Métrica':<25} {'Baseline':>15} {'Mejorado':>15}")
print("="*62)
for metric, (base_val, imp_val) in metrics.items():
    delta = imp_val - base_val
    arrow = "↑" if delta > 0 else "↓"
    # For gap, lower is better
    if 'Gap' in metric:
        arrow = "↓" if delta < 0 else "↑"
    print(f"{metric:<25} {base_val:>15.4f} {imp_val:>15.4f}  {arrow} {abs(delta):.4f}")

# Gráfico de barras comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparación: Baseline vs. Modelo Mejorado", fontsize=13, fontweight='bold')

metric_names = ['Val Accuracy', 'Val F1 (macro)']
base_vals = [scores_base['test_accuracy'].mean(), scores_base['test_f1_macro'].mean()]
base_stds = [scores_base['test_accuracy'].std(), scores_base['test_f1_macro'].std()]
imp_vals  = [scores_improved['test_accuracy'].mean(), scores_improved['test_f1_macro'].mean()]
imp_stds  = [scores_improved['test_accuracy'].std(), scores_improved['test_f1_macro'].std()]

x = np.arange(len(metric_names))
width = 0.35
axes[0].bar(x - width/2, base_vals, width, yerr=base_stds, label='Baseline', color='#e74c3c', alpha=0.8, capsize=5)
axes[0].bar(x + width/2, imp_vals, width, yerr=imp_stds, label='Mejorado', color='#2ecc71', alpha=0.8, capsize=5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Score")
axes[0].set_title("Métricas de validación (CV 5-fold)")
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gap train-val
gap_base = scores_base['train_accuracy'].mean() - scores_base['test_accuracy'].mean()
gap_imp  = scores_improved['train_accuracy'].mean() - scores_improved['test_accuracy'].mean()
axes[1].bar(['Baseline', 'Mejorado'], [gap_base, gap_imp],
            color=['#e74c3c', '#2ecc71'], alpha=0.8, width=0.4)
axes[1].set_title("Gap Train-Val Accuracy\n(menor = menos overfitting)")
axes[1].set_ylabel("Gap")
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate([gap_base, gap_imp]):
    axes[1].text(i, v + 0.005, f"{v:.4f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("viz3_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


---
# PARTE 3: Análisis Crítico (30%)

Esta sección evalúa tu capacidad de interpretar resultados y pensar más allá del código.

### 3.1 Feature Importance

Usando tu modelo (o el baseline), extrae y visualiza la importancia de cada feature. ¿Coincide con lo que observaste en la exploración?

In [ ]:
# ============================================================
# 3.1 Feature Importance
# ============================================================

importances = rf_improved.feature_importances_
feat_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=True)

color_map = {
    'CSF': '#3498db',
    'PLASMA': '#e67e22',
    'AGE': '#9b59b6',
    'SEX': '#9b59b6',
    'EDUCATION': '#9b59b6'
}

colors = []
for feat in feat_importance['Feature']:
    if feat.startswith('CSF'):
        colors.append('#3498db')
    elif feat.startswith('PLASMA'):
        colors.append('#e67e22')
    else:
        colors.append('#9b59b6')

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(feat_importance['Feature'], feat_importance['Importance'], color=colors, alpha=0.85)
ax.set_xlabel("Importancia (Mean Decrease in Impurity)", fontsize=11)
ax.set_title("Feature Importance — Modelo Mejorado\n(Random Forest regularizado)", fontsize=12, fontweight='bold')

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Biomarcadores CSF'),
    Patch(facecolor='#e67e22', label='Biomarcadores Plasma'),
    Patch(facecolor='#9b59b6', label='Variables demográficas'),
]
ax.legend(handles=legend_elements, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("viz4_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

# Top 5 features
print("\nTop 5 features mas importantes:")
print(feat_importance.tail(5)[['Feature', 'Importance']].iloc[::-1].to_string(index=False))

print("CSF_AB42, CSF_TAU/PTAU y los ratios de plasma dominan, consistente con la EDA.")
print("Los ratios derivados (PTAU217/AB42) capturan información combinada y destacan sobre features individuales.")


### 3.2 Análisis de errores

Genera una **matriz de confusión** de tu modelo y analiza:
- ¿Qué clase es más difícil de clasificar?
- ¿Qué tipo de errores son más frecuentes (falsos positivos vs. falsos negativos)?
- ¿Tiene implicaciones clínicas el tipo de error más frecuente?

In [ ]:
# ============================================================
# 3.2 Análisis de errores
# ============================================================
from sklearn.model_selection import cross_val_predict

y_pred_cv = cross_val_predict(rf_improved, X, y, cv=cv, method='predict')
class_names = le.classes_  # ['AD', 'MCI', 'NC'] ordenado por LabelEncoder

print("=== Reporte de clasificación (CV 5-fold, out-of-fold) ===")
print(classification_report(y, y_pred_cv, target_names=class_names))

cm = confusion_matrix(y, y_pred_cv)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title("Matriz de confusión (conteos)", fontweight='bold')

cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm.round(2), display_labels=class_names)
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title("Matriz de confusión (normalizada por clase)", fontweight='bold')

plt.tight_layout()
plt.savefig("viz5_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Análisis de errores ===")
print("""
- CLASE MÁS DIFÍCIL: MCI presenta el recall más bajo. Es la clase más confundida con NC y AD.

  • falsos negativos de MCI: el modelo clasifica pacientes en deterioro leve como normales.
  • falsos positivos de AD: sobrediagnóstico de Alzheimer en pacientes aún en etapa temprana.

  El error más peligroso en este contexto es clasificar AD como NC (falso negativo de AD):
  el paciente no recibirá tratamiento ni seguimiento, permitiendo progresión sin intervención.
      
  Clasificar NC como AD (falso positivo) genera ansiedad, estigma y costos innecesarios, pero
  es menos grave que el escenario anterior.
  En screening clinico, se priorizaría maximizar el recall de AD a costa de más falsos positivos.
""")


### 3.3 Preguntas abiertas finales

Responde las siguientes preguntas en la celda de abajo:

1. **¿Qué problemas específicos identificaste en el modelo baseline y qué cambios hiciste para abordarlos?**

2. **Si tuvieras acceso a variables clínicas adicionales (ej: MMSE, CDR, ApoE genotype), ¿cuáles incluirías y por qué?**

3. **En un contexto clínico real, ¿qué tipo de error sería más peligroso: clasificar a un paciente con AD como NC, o clasificar a un paciente NC como AD? ¿Cómo ajustarías tu modelo para reflejar esto?**

4. **¿Qué harías diferente si tuvieras más tiempo? Describe al menos dos mejoras concretas.**

El baseline presentaba cuatro problemas principales:

- Sin límite de profundidad (`max_depth=None`): Los abroles crecen hasta memorizar el conjunto de entrenamiento. Se corrigió con `max_depth=6`.
- `min_samples_split=2` y `min_samples_leaf=1`: Permiten splits con una sola muestra, generando hojas perfectamente puras que no generalizan. Se cambió a 10 y 5 respectivamente.
- `max_features=None`: Usar todas las features en cada split elimina la randomness que hace al Random Forest robusto. Se cambió a `'sqrt'`.
- `class_weight=None`: Ignora el desbalance de clases, sesgando el modelo hacia las clases mayoritarias. Se corrigió con `class_weight='balanced'`.
- Evaluación solo por CV sin reporte de train scores explícito: El baseline calculaba ambos, pero el gap enorme (train ≈ 1.0, val ≈ 0.7x) significa overfitting severo.

---
incluiria: 
- MMSE (Mini-Mental State Examination): Prueba cognitiva global. Alto poder predictivo, especialmente para distinguir NC de MCI y AD leve.
- CDR (Clinical Dementia Rating): Escala semiestructurada que caracteriza la severidad del deterioro. Es el estándar clínico para estadificar MCI y AD.
- ApoE genotype (ε4): El alelo ε4 del gen APOE es el principal factor de riesgo genético para AD esporádico. Su presencia multiplica el riesgo y potencia la señal de los biomarcadores.
- Volumen hipocampal (MRI): La atrofia hipocampal es el marcador estructural por excelencia en AD. Complementa los marcadores de CSF con información de neuroimagen.
- velocidad de procesamiento cognitivo: Tests neuropsicológicos adicionales (TMT, fluencia verbal) que capturan deterioro antes del MMSE.

---


Error mas peligroso:

Clasificar a un paciente con **AD como NC (falso negativo de AD)** es el error más peligroso, porque el paciente no recibe diagnóstico ni tratamiento temprano; se pierde la ventana de intervención y el daño neurológico progresa sin monitoreo.

Para mitigarlo, se ajustaría el modelo con un umbral de decisión más bajo para la clase AD: en lugar de decidir por la clase con mayor probabilidad, se clasifica como AD si la probabilidad supera ~0.3 en vez de 0.5. Adema scomo métrica de optimización durante la búsqueda de hiperparámetros (en vez de accuracy o F1 macro). Finalmente se mitiga con penalización mayor para la clase AD para que los errores en esa clase tengan mayor costo durante el entrenamiento.

---

Mejoras:

Optimización de hiperparámetros con Optuna o GridSearchCV: Explorar sistemáticamente combinaciones de hiperparámetros optimizando recall de AD. Se podría combinar con selección de features (RFE, SHAP-based) para reducir ruido.

Ensamble heterogéneo (stacking): Combinar Random Forest, XGBoost y Logistic Regression como modelos base, con un meta-clasificador que aprenda a combinar sus predicciones. Esto reduce la varianza sin aumentar el sesgo, y es especialmente útil cuando diferentes modelos capturan aspectos distintos de los datos (lineal vs. no-lineal).

Análisis de SHAP values: Más allá de la importancia por impureza, SHAP permite interpretar la contribución de cada feature para cada paciente individual, lo que es crítico en contextos clínicos donde se necesita explicabilidad.


---
## Entrega

1. Asegúrate de que todo el notebook corre de principio a fin sin errores.
2. Haz commit y push a tu fork.
3. Envía la liga de tu repositorio a Angel Peña (6271506213).

**¡GenoBit te da las gracias por participar!** 